# VideoMAE boundary detector — Colab runner

Fine-tunes VideoMAE on the step-boundary task and evaluates it with this
project's own segmentation metrics (`peaks_from_score`, boundary F1,
leave-one-clip-out).

**Before you start:** Runtime → Change runtime type → **GPU (T4)**.

Run the cells in order. Cell 2 is the only one you edit.

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch, sys
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > GPU'

## 2. Settings — **edit these two lines**

`DRIVE_VIDEOS` is the Drive folder holding your five `.mp4` files.
`REPO` is your GitHub repo (public = no token needed).

In [ ]:
REPO = 'https://github.com/MostafaTaha04/Split-Video-Into-Actions-Project.git'
DRIVE_VIDEOS = '/content/drive/MyDrive/split-video-data'   # <- your .mp4 files live here

## 3. Mount Drive (for the videos)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
assert os.path.isdir(DRIVE_VIDEOS), f'Not found: {DRIVE_VIDEOS}'
print(sorted(os.listdir(DRIVE_VIDEOS)))

## 4. Get the code

**Re-run this cell every time I push a change** — it pulls the latest version.

In [ ]:
import os, subprocess
if os.path.isdir('/content/proj/.git'):
    print(subprocess.run(['git','-C','/content/proj','pull','--ff-only'],
                         capture_output=True, text=True).stdout)
else:
    !git clone -q $REPO /content/proj
%cd /content/proj
!git log --oneline -1

## 5. Install dependencies (~2 min, once per session)

In [ ]:
!pip install -q transformers decord av
import transformers; print('transformers', transformers.__version__)

## 6. Link the videos into the repo

The `.mp4` files are gitignored, so they come from Drive. Symlinks avoid
copying 290 MB.

In [ ]:
import os, glob
os.chdir('/content/proj')
for src in glob.glob(os.path.join(DRIVE_VIDEOS, '*.mp4')):
    dst = os.path.join('/content/proj', os.path.basename(src))
    if not os.path.exists(dst):
        os.symlink(src, dst)
print([os.path.basename(p) for p in glob.glob('/content/proj/*.mp4')])

## 7. Smoke test (~2 min)

One fold, six training steps. Proves the whole path works before the long run.
**The numbers are meaningless** — we only care that it finishes.

In [ ]:
!python videomae_boundary.py --smoke

## 8. Full run (~45–90 min on a T4)

Leave-one-clip-out over all five clips. If Colab disconnects, lower `--epochs`
to 2 or run fewer folds with `--folds 3`.

In [ ]:
!python videomae_boundary.py --epochs 3 --batch-size 2 --out videomae_results.json

## 9. Show the results — **paste this output back into the chat**

In [ ]:
import json
r = json.load(open('/content/proj/videomae_results.json'))
print(json.dumps(r, indent=2))

## 10. Save results back to Drive

So they survive the runtime being recycled. Then paste the JSON above into the
chat and I'll integrate it into the report.

In [ ]:
import shutil
shutil.copy('/content/proj/videomae_results.json', DRIVE_VIDEOS + '/videomae_results.json')
print('saved to Drive')